# 01. Previsão de Desmatamento

**Objetivo:** Prever municípios com maior probabilidade de desmatamento para alocação eficiente de recursos de fiscalização.

**Impacto no Negócio:** Otimizar alocação de recursos de fiscalização ambiental, focando em municípios com maior probabilidade de desmatamento futuro.

**Dados de Entrada:** `data/04_modelagem/dataset_preditivo_com_precos.parquet`

**Dados de Saída:** `data/03_gold/ranking_risco_desmatamento_2023.parquet`

In [1]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

📤 Ambiente Google Colab detectado
Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive montado em /content/drive


In [2]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE (COMPARTILHADO)
# ============================================================================

import sys
import os
from pathlib import Path

# Adicionar caminho do módulo compartilhado
sys.path.append(os.path.join(os.getcwd(), '..'))

# Importar módulo compartilhado com tipos customizados
from src.utils.predicao import (
    detectar_ambiente_colab,
    montar_google_drive,
    configurar_caminhos_dados,
    UFS_AMAZONIA_LEGAL,
    ANO_LIMITE_TREINO,
    ANO_TESTE,
    ANO_PREVISAO
)

# Importar tipos customizados
from src.utils import (
    CodigoIBGE,
    Ano,
    Probabilidade,
    DataFrame,
    CaminhoArquivo,
    MetricasAvaliacao,
    ValidadorTipo
)

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Caminho do arquivo no Google Drive
CAMINHO_ARQUIVO_DRIVE: CaminhoArquivo = '/content/drive/MyDrive/dados_analise/dataset_preditivo_com_precos.parquet'

# Configurar caminhos baseado no ambiente
caminhos_configurados = configurar_caminhos_dados(
    caminho_arquivo_drive=CAMINHO_ARQUIVO_DRIVE,
    caminho_dados_local='../data/04_modelagem/dataset_preditivo_com_precos.parquet',
    caminho_saida_local='../data/03_gold/',
    nome_arquivo_saida='ranking_risco_desmatamento_2023.parquet'
)

CAMINHO_DADOS: CaminhoArquivo = caminhos_configurados['caminho_dados']
CAMINHO_SAIDA: CaminhoArquivo = caminhos_configurados['caminho_saida']

print(f"\nCaminho dos dados: {CAMINHO_DADOS}")
print(f"Caminho de saída: {CAMINHO_SAIDA}")

ModuleNotFoundError: No module named 'src'

In [ ]:
## 1. Configuração e Importações
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Importar funções do módulo compartilhado com tipos
from src.utils.predicao import (
    carregar_dados,
    filtrar_amazonia_legal,
    preparar_features_modelo,
    dividir_dados_temporalmente,
    calcular_pesos_classes,
    treinar_modelo_random_forest,
    avaliar_modelo
)

# Importar tipos customizados
from src.utils import (
    CodigoIBGE,
    Ano,
    Probabilidade,
    DataFrame,
    MetricasAvaliacao,
    ValidadorTipo
)

# ============================================================================
# CONFIGURAÇÃO
# ============================================================================

# Features selecionadas para o modelo (baseadas em análise de importância)
FEATURES_MODELO = [
    'cod_ibge', 'ano', 'vab_agro_mil_reais', 'ppm_bovinos_cabecas', 
    'num_embargos', 'idhm', 'precipitacao_total_mm', 'precipitacao_media_diaria_mm',
    'estacao_chuva', 'anos_obs', 'log_bovinos', 'log_vab',
    'pressao_economica', 'preco_boi_gordo_rs', 'preco_milho_rs', 'preco_soja_rs',
    'producao_soja_mil_ton', 'producao_milho_mil_ton', 'pressao_agro_alta',
    'indice_pressao_preco'
]

TARGET_DESMATAMENTO = 'tem_desmatamento'

In [ ]:
## 2. Funções Auxiliares

#Esta seção contém funções reutilizáveis para carregamento, preparação e validação de dados.

In [ ]:
# ============================================================================
# PASSO 2: FILTRAGEM DA AMAZÔNIA LEGAL
# ============================================================================

print("\nPASSO 2: Filtrando dados para Amazônia Legal...")

# Filtra apenas municípios da Amazônia Legal
df_amazonia = filtrar_amazonia_legal(df_completo, UFS_AMAZONIA_LEGAL)

print(f"✓ Amazônia Legal: {df_amazonia.shape[0]:,} observações")
print(f"✓ Municípios na Amazônia Legal: {df_amazonia['cod_ibge'].nunique():,}")

# Estatísticas de desmatamento
municipios_com_desmatamento = df_amazonia['tem_desmatamento'].sum()
percentual_desmatamento = df_amazonia['tem_desmatamento'].mean() * 100
area_total_desmatada = df_amazonia['area_desmatada_ha'].sum()

print(f"\n📊 ESTATÍSTICAS DE DESMATAMENTO:")
print(f"  Municípios com desmatamento: {municipios_com_desmatamento:,} ({percentual_desmatamento:.1f}%)")
print(f"  Área total desmatada: {area_total_desmatada:,.0f} ha")

In [ ]:
# ============================================================================
# PASSO 3: PREPARAÇÃO DAS FEATURES
# ============================================================================

print("\nPASSO 3: Preparando features para o modelo...")

# Prepara features e target
TARGET_DESMATAMENTO = 'tem_desmatamento'
X, y = preparar_features_modelo(df_amazonia, FEATURES_MODELO, TARGET_DESMATAMENTO)

print(f"✓ Dataset preparado: {X.shape[0]:,} observações × {X.shape[1]} features")
print(f"✓ Target: {TARGET_DESMATAMENTO}")
print(f"✓ Distribuição target: {y.mean()*100:.2f}% positivos")

In [ ]:
# ============================================================================
# PASSO 4: DIVISÃO TEMPORAL DOS DADOS
# ============================================================================

print("\nPASSO 4: Dividindo dados temporalmente (treino/teste)...")

# Divide os dados em treino e teste baseado em critério temporal
X_train, X_test, y_train, y_test = dividir_dados_temporalmente(
    X, y, ANO_LIMITE_TREINO, ANO_TESTE
)

print(f"✓ Treino: {X_train.shape[0]:,} observações ({X_train['ano'].min()}-{X_train['ano'].max()})")
print(f"✓ Teste: {X_test.shape[0]:,} observações ({X_test['ano'].min()}-{X_test['ano'].max()})")
print(f"✓ Distribuição treino: {y_train.mean()*100:.2f}% positivos")
print(f"✓ Distribuição teste: {y_test.mean()*100:.2f}% positivos")

In [ ]:
# ============================================================================
# PASSO 5: CÁLCULO DE PESOS DAS CLASSES
# ============================================================================

print("\nPASSO 5: Calculando pesos das classes para lidar com desbalanceamento...")

# Calcula pesos para lidar com desbalanceamento de classes
pesos_classes = calcular_pesos_classes(y_train)

print(f"✓ Pesos calculados:")
print(f"  Classe 0 (sem desmatamento): {pesos_classes[0]:.4f}")
print(f"  Classe 1 (com desmatamento): {pesos_classes[1]:.4f}")

In [ ]:
# ============================================================================
# PASSO 6: TREINAMENTO DO MODELO
# ============================================================================

print("\nPASSO 6: Treinando modelo Random Forest...")

# Treina o modelo com os parâmetros otimizados
modelo_desmatamento = treinar_modelo_random_forest(X_train, y_train, pesos_classes)

print("✓ Modelo treinado com sucesso!")
print(f"✓ Tipo: RandomForestClassifier")
print(f"✓ Estimators: 200")
print(f"✓ Max Depth: 6 (para evitar overfitting)")

In [ ]:
# ============================================================================
# PASSO 7: AVALIAÇÃO DO MODELO
# ============================================================================

print("\nPASSO 7: Avaliando desempenho do modelo...")

# Gera previsões probabilísticas
y_pred_proba = modelo_desmatamento.predict_proba(X_test)[:, 1]

# Encontra threshold ótimo para maximizar F1-score
threshold_otimo, precision, recall, f1_scores = otimizar_threshold(y_test, y_pred_proba)

# Aplica threshold otimizado
y_pred_otimizado = (y_pred_proba >= threshold_otimo).astype(int)

# Calcula métricas de avaliação
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

print("✓ MÉTRICAS DE AVALIAÇÃO:")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  Precision-Recall AUC: {pr_auc:.4f}")
print(f"  Threshold ótimo: {threshold_otimo:.4f}")

print("\n✓ RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(y_test, y_pred_otimizado, 
                          target_names=['Sem Desmatamento', 'Com Desmatamento']))

In [ ]:
# ============================================================================
# PASSO 8: ANÁLISE DE IMPORTÂNCIA DAS FEATURES
# ============================================================================

print("\nPASSO 8: Analisando importância das features...")

# Cria DataFrame com importância das features
df_importancia = criar_feature_importance(modelo_desmatamento, FEATURES_MODELO)

print("✓ TOP 10 FEATURES MAIS IMPORTANTES:")
print(df_importancia.head(10).to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 9: GERAÇÃO DE PREVISÕES PARA 2023
# ============================================================================

print("\nPASSO 9: Gerando previsões para 2023...")

# Gera previsões para 2023 usando dados de 2022 como base
df_previsoes = gerar_previsoes_ano_seguinte(
    df_amazonia, 
    modelo_desmatamento, 
    FEATURES_MODELO, 
    ANO_TESTE, 
    ANO_PREVISAO
)

print(f"✓ Previsões geradas para {len(df_previsoes)} observações")
print(f"✓ Base: dados de {ANO_TESTE}")
print(f"✓ Previsão para: {ANO_PREVISAO}")

In [ ]:
# ============================================================================
# PASSO 10: CRIAÇÃO DO RANKING DE MUNICÍPIOS
# ============================================================================

print("\nPASSO 10: Criando ranking de municípios por risco...")

# Cria ranking de municípios baseado em probabilidade
COLUNA_PROBABILIDADE = f'probabilidade_desmatamento_{ANO_PREVISAO}'
COLUNAS_RANKING = ['cod_ibge', 'municipio', 'uf', 'area_desmatada_ha', 'vab_agro_mil_reais']

ranking_municipios = criar_ranking_municipios(
    df_previsoes, 
    COLUNA_PROBABILIDADE, 
    COLUNAS_RANKING
)

print(f"✓ Ranking criado: {len(ranking_municipios)} municípios únicos")
print(f"\n✓ TOP 20 MUNICÍPIOS COM MAIOR PROBABILIDADE DE DESMATAMENTO EM {ANO_PREVISAO}:")
print(ranking_municipios.head(20).to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 11: CÁLCULO DE ESTATÍSTICAS DE IMPACTO
# ============================================================================

print("\nPASSO 11: Calculando estatísticas de impacto...")

# Calcula estatísticas para os top 50 municípios
TOP_MUNICIPIOS_ANALISE = 50
estatisticas_impacto = calcular_estatisticas_impacto(
    ranking_municipios,
    COLUNA_PROBABILIDADE,
    'area_desmatada_ha',
    'vab_agro_mil_reais',
    TOP_MUNICIPIOS_ANALISE
)

print("✓ ESTATÍSTICAS DE IMPACTO (TOP 50 MUNICÍPIOS):")
print(f"  Quantidade de municípios: {estatisticas_impacto['quantidade_municipios']}")
print(f"  Probabilidade média de desmatamento: {estatisticas_impacto['probabilidade_media']*100:.1f}%")
print(f"  Área desmatada histórica: {estatisticas_impacto['area_total']:,.0f} ha")
print(f"  VAB agropecuário total: R$ {estatisticas_impacto['vab_total']*1000:,.0f}")

In [ ]:
# ============================================================================
# PASSO 12: SALVAMENTO DOS RESULTADOS
# ============================================================================

print("\nPASSO 12: Salvando resultados...")

# Salva o ranking de risco
salvar_ranking(ranking_municipios, CAMINHO_SAIDA)

print(f"✓ Ranking salvo em: {CAMINHO_SAIDA}")
print(f"✓ Total de municípios: {len(ranking_municipios)}")

print("\n" + "="*70)
print("✅ ANÁLISE PREDITIVA DE DESMATAMENTO CONCLUÍDA COM SUCESSO")
print("="*70)

In [ ]:
# Célula removida - código antigo não usado

# Célula removida - código antigo não usado

In [ ]:
# Célula removida - código antigo não usado

# Célula removida - código antigo não usado

In [ ]:
# Célula removida - código antigo não usado

## 11. Conclusão

**Resumo da Análise:**
- Modelo de Random Forest treinado com métricas de avaliação calculadas
- Municípios classificados por probabilidade de desmatamento
- Top 50 municípios identificados para fiscalização prioritária

**Impacto no Negócio:**
- Alocação eficiente de recursos de fiscalização
- Foco em municípios com maior probabilidade de desmatamento futuro
- Base para sistema de alertas automáticos

**Próximos Passos:**
- Integrar com sistemas de monitoramento satelital
- Validar previsões com dados de 2023 quando disponíveis
- Implementar sistema de alertas automáticos